<a href="https://colab.research.google.com/github/KwancheolShin/introduction-to-deep-learning/blob/main/%ED%8C%8C%EC%9D%B4%EC%8D%AC%EA%B8%B0%EC%B4%884.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 파이썬 기초 4: 객체와 참조, 복사, 저장, 명령줄 인자

딥러닝 코드를 읽다 보면 다음과 같은 질문을 만나게 된다.

* `params[key] -= lr * grads[key]`를 `params[key] = params[key] - lr * grads[key]`로 바꾸면 왜 학습이 안 될까?
* `copy.copy`, `copy.deepcopy`, `arr.copy()`는 무엇이 다를까?
* `keepdims=True`는 언제 쓸까?
* `f"{acc:.4f}"`의 `:.4f`는 무엇일까?
* 학습한 결과를 파일로 저장하려면? `python train.py --lr 1e-3`은 무슨 뜻일까?

이 노트북은 이 질문들에 답하는 데 필요한 파이썬의 기본 개념을 정리한다.

| 절 | 내용 |
|---|---|
| 17 | 객체, 변수, 참조 |
| 18 | 가변 객체와 불변 객체 |
| 19 | 재바인딩과 in-place 변경 |
| 20 | 복사: `=`, `copy.copy`, `copy.deepcopy`, `arr.copy()` |
| 21 | numpy의 `axis`와 `keepdims` |
| 22 | 문자열 더 알아보기: f-string 서식, `r"..."`, `b"..."` |
| 23 | `zip`과 unpacking 더 알아보기 |
| 24 | 객체지향 프로그래밍 맛보기: 클래스와 상속 |
| 25 | 저장하고 불러오기: `json`, `pickle`, `np.save`, `torch.save` |
| 26 | 명령줄 인자: `sys.argv`, `argparse` |
| 27 | 스크립트 실행과 노트북(커널) 실행 |
| 28 | 연습문제 |

# 17. 객체, 변수, 참조

## 17.1. 변수는 상자가 아니라 이름표이다
파이썬에서 `a = [1, 2, 3]`을 실행하면

1. 메모리 어딘가에 list **객체(object)** 가 만들어지고,
2. `a`라는 **이름(name)** 이 그 객체를 **가리키게(참조, reference)** 된다.

객체는 메모리의 한 덩어리로, **타입**(list, int, ...)과 **값**을 가진다. 변수는 값을 담는 상자가 아니라, 객체에 붙인 **이름표**이다.
이름을 객체에 연결하는 것을 **바인딩(binding)** 이라고 한다.

`b = a`는 값을 복사하지 않는다. **같은 객체에 이름표를 하나 더 붙일 뿐이다.**

```
a ──┐
    ├──▶ [1, 2, 3]   (list 객체 하나)
b ──┘
```

In [1]:
a = [1, 2, 3]
b = a                  # 복사가 아니라 같은 객체에 이름을 하나 더 붙인다
b.append(4)            # b를 통해 객체를 바꾸면
print("a =", a)        # a에서도 바뀐 것이 보인다
print("b =", b)

a = [1, 2, 3, 4]
b = [1, 2, 3, 4]


## 17.2. `id()`, `is`, `==`
* `id(x)`: 객체마다 고유한 번호. CPython에서는 객체의 메모리 주소이다.
* `x is y`: 두 이름이 **같은 객체**를 가리키는가? (동일성, identity) &nbsp; `id(x) == id(y)`와 같다.
* `x == y`: 두 객체의 **값이 같은가?** (동등성, equality)

In [2]:
a = [1, 2, 3]
b = a
c = [1, 2, 3]          # 값은 같지만 새로 만든 다른 객체

print(id(a), id(b), id(c))
print("a is b:", a is b, "  a == b:", a == b)
print("a is c:", a is c, "  a == c:", a == c)

1518927535744 1518927535744 1518927536384
a is b: True   a == b: True
a is c: False   a == c: True


## 17.3. dict의 칸에도 참조가 들어 있다
dict는 **key로 value를 찾는 표**이다. 엄밀하게 말하면 각 칸에는 value 객체 자체가 아니라 **value 객체를 가리키는 참조**가 들어 있다.

* `d[key]`로 읽는다. key가 없으면 `KeyError`가 난다.
* `d[key] = 값`: key가 **없으면 새 칸을 만들고, 있으면 그 칸의 참조를 새 객체로 바꾼다.** 같은 key가 두 개가 되는 일은 없다.
* key는 중복될 수 없고, 바꿀 수 없는 객체(문자열, 숫자, tuple)만 key가 될 수 있다. value는 무엇이든 된다.

```
 params (dict 객체)                     메모리 어딘가
┌──────────────────────┐
│ 'W1' : ───────────────┼──────▶  배열 [[...]]
│ 'b1' : ───────────────┼──────▶  배열 [...]
└──────────────────────┘
```

In [3]:
import numpy as np

params = {"W1": np.array([1.0, 2.0]), "b1": np.array([0.0])}
w = params["W1"]                       # dict 칸이 가리키는 배열에 이름 w를 하나 더 붙인다
print("w is params['W1']:", w is params["W1"])

params["b1"] = np.array([9.0])         # 'b1' 칸의 참조를 새 배열로 바꾼다 (칸은 여전히 하나)
print(params)
print("keys:", list(params.keys()))

w is params['W1']: True
{'W1': array([1., 2.]), 'b1': array([9.])}
keys: ['W1', 'b1']


# 18. 가변 객체와 불변 객체

| 불변 (immutable): 만든 뒤 값을 바꿀 수 없음 | 가변 (mutable): 만든 뒤 값을 바꿀 수 있음 |
|---|---|
| `int`, `float`, `bool`, `str`, `tuple` | `list`, `dict`, `set`, numpy 배열, torch tensor, 직접 만든 클래스의 객체 |

바꿀 수 없는 것은 **객체**이다. 변수(이름표)는 언제든 다른 객체로 바꿔 붙일 수 있다. 그래서 `x = 5` 다음에 `x = 6`을 쓸 수 있다.

## 18.1. `+=`의 동작은 객체에 따라 다르다
* 불변 객체(`int`): 값을 고칠 수 없으므로 **새 객체**를 만들고 이름표를 옮긴다. `id`가 바뀐다.
* 가변 객체(`list`): **그 객체 자체**를 고친다. `id`가 그대로이다.

In [4]:
x = 1000
print("int  before:", id(x))
x += 1
print("int  after :", id(x), "-> 다른 객체 (새 객체 1001)")

L = [1, 2]
print("list before:", id(L))
L += [3]
print("list after :", id(L), "-> 같은 객체,", L)

int  before: 1518927322960
int  after : 1518929874608 -> 다른 객체 (새 객체 1001)
list before: 1518930798400
list after : 1518930798400 -> 같은 객체, [1, 2, 3]


## 18.2. 문자열과 tuple은 안의 값을 바꿀 수 없다

In [5]:
s = "abc"
try:
    s[0] = "z"
except TypeError as e:
    print("str  :", e)
print("s.upper() =", s.upper(), " / s =", s, "(upper()는 새 문자열을 돌려준다)")

t = (1, 2)
try:
    t[0] = 9
except TypeError as e:
    print("tuple:", e)

t2 = (1, [2, 3])       # tuple이 바꿀 수 없는 것은 자기 칸의 참조뿐이다
t2[1].append(4)        # 칸이 가리키는 list는 가변이므로 바뀔 수 있다
print("t2 =", t2)

str  : 'str' object does not support item assignment
s.upper() = ABC  / s = abc (upper()는 새 문자열을 돌려준다)
tuple: 'tuple' object does not support item assignment
t2 = (1, [2, 3, 4])


## 18.3. 아무도 가리키지 않는 객체는?
파이썬은 객체마다 자신을 가리키는 참조의 개수(**reference count**)를 세어 둔다. 이 수가 0이 되면 객체는 메모리에서 지워진다.
서로를 가리키는 객체들은 가끔 도는 **garbage collector**가 따로 정리한다.

예외: CPython은 자주 쓰는 작은 정수(−5부터 256까지)를 미리 만들어 두고 끝까지 지우지 않는다. 그래서 작은 정수는 어디서 만들어도 같은 객체이다.
(이것은 파이썬 내부 사정이므로 코드를 짤 때 기대해서는 안 된다. 값 비교는 항상 `==`로 한다.)

In [6]:
a = 5; b = 5
print("5 is 5   :", a is b)        # 미리 만들어 둔 같은 객체

a = int("1000"); b = int("1000")
print("1000 is 1000:", a is b, "  1000 == 1000:", a == b)

5 is 5   : True
1000 is 1000: False   1000 == 1000: True


**dict의 key와 불변 객체.** key가 중간에 바뀌면 칸을 찾을 수 없게 되므로, dict의 key는 불변 객체여야 한다. 그래서 문자열과 tuple은 key가 될 수 있고 list는 될 수 없다.

In [7]:
d = {("lr", 0.01): 0.97}          # tuple은 key가 될 수 있다
try:
    d[["lr", 0.01]] = 0.97         # list는 안 된다
except TypeError as e:
    print(e)

unhashable type: 'list'


# 19. 재바인딩과 in-place 변경

같은 계산처럼 보이는 두 줄이 다르게 동작한다.

| 코드 | 하는 일 |
|---|---|
| `a = a - 1` | 오른쪽에서 **새 배열**을 만들고, 이름 `a`를 새 배열로 옮긴다 (**재바인딩, rebinding**) |
| `a -= 1` | numpy 배열이면 **그 배열의 값을 그 자리에서** 바꾼다 (**in-place 변경, mutation**) |

같은 배열을 다른 이름이 함께 가리키고 있을 때 차이가 드러난다.

In [8]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])
b = a                      # a와 b는 같은 배열을 가리킨다

a -= 1                     # in-place: 같은 배열의 값을 바꾼다
print("a -= 1    ->  a =", a, " b =", b, " same object:", a is b)

a = a - 1                  # rebinding: 새 배열을 만들어 a만 옮긴다
print("a = a - 1 ->  a =", a, " b =", b, " same object:", a is b)

a -= 1    ->  a = [0. 1. 2.]  b = [0. 1. 2.]  same object: True
a = a - 1 ->  a = [-1.  0.  1.]  b = [0. 1. 2.]  same object: False


## 19.1. 딥러닝 코드에서: parameter와 층이 같은 배열을 공유할 때
NumPy로 만든 `MultiLayerNet`에서는 `params["W1"]`과 `Affine` 층 안의 `self.W`가 **같은 배열**을 가리킨다.
순전파는 층의 `self.W`로 계산하고, 갱신은 `params`를 통해서 한다.

In [9]:
class Affine:
    def __init__(self, W):
        self.W = W                     # 복사하지 않는다: params["W1"]과 같은 배열을 가리킨다

params = {"W1": np.array([1.0, 2.0, 3.0])}
layer = Affine(params["W1"])
print("same array:", layer.W is params["W1"])

params["W1"] -= 0.5                    # in-place 갱신: 층에도 반영된다
print("after -= :", params["W1"], layer.W)

params["W1"] = params["W1"] - 0.5      # 재바인딩: params만 새 배열로 바뀌고 층은 옛 배열을 본다
print("after =  :", params["W1"], layer.W, "  same array:", layer.W is params["W1"])

same array: True
after -= : [0.5 1.5 2.5] [0.5 1.5 2.5]
after =  : [0. 1. 2.] [0.5 1.5 2.5]   same array: False


그래서 NumPy 학습 코드는 `params[key] -= lr * grads[key]`처럼 **in-place로** 갱신한다.
`=`로 갱신하면 층이 계속 처음 가중치로 계산하므로 학습이 전혀 되지 않는다.
PyTorch의 `optimizer.step()`도 내부에서 in-place 연산(`p.add_(...)`처럼 이름이 `_`로 끝나는 연산)으로 parameter를 바꾼다.

## 19.2. 함수에 인자를 넘길 때도 참조가 넘어간다
함수를 부르면 인자로 준 객체의 **참조**가 매개변수에 바인딩된다 (call by sharing).
그래서 함수 안에서 가변 객체를 고치면 함수 밖에서도 바뀐 것이 보인다. 매개변수를 재바인딩하는 것은 밖에 영향을 주지 않는다.

In [10]:
def add_item(L):
    L.append(99)          # 객체를 고친다 -> 밖에서도 보인다

def rebind(L):
    L = [0, 0, 0]         # 매개변수 이름만 새 객체로 옮긴다 -> 밖에는 영향 없음

data = [1, 2]
add_item(data); print("after add_item:", data)
rebind(data);   print("after rebind  :", data)

after add_item: [1, 2, 99]
after rebind  : [1, 2, 99]


# 20. 복사: `=`, `copy.copy`, `copy.deepcopy`, `arr.copy()`

**이름은 비슷하지만 서로 다른 것**이 있으니 주의하자. 점(`.`) 앞에 무엇이 오는지 보면 구별된다.

| 코드 | 무엇 | 하는 일 |
|---|---|---|
| `b = a` | 대입 | 복사하지 않는다. 같은 객체에 이름을 하나 더 붙인다 |
| `copy.copy(a)` | `copy` **모듈**의 함수 | **얕은 복사(shallow copy)**: 바깥 컨테이너만 새로 만들고, 안의 객체는 참조를 공유한다 |
| `copy.deepcopy(a)` | `copy` **모듈**의 함수 | **깊은 복사(deep copy)**: 안의 객체까지 모두 새로 만든다 |
| `arr.copy()` | numpy **배열 자신**의 메서드 | 새 메모리를 잡아 숫자 값을 모두 복사한다 |
| `L.copy()`, `d.copy()` | list, dict 자신의 메서드 | 얕은 복사 (`copy.copy`와 같다) |
| `t.clone()` | torch tensor의 메서드 | 값을 모두 복사한다 |

## 20.1. 배열이 든 dict를 복사하면

In [11]:
import copy
import numpy as np

params  = {"W1": np.array([1.0, 2.0]), "b1": np.array([0.0])}
alias   = params                     # 대입
shallow = copy.copy(params)          # 얕은 복사
deep    = copy.deepcopy(params)      # 깊은 복사

print("dict 자체가 같은 객체?  ", alias is params, shallow is params, deep is params)
print("안의 W1이 같은 배열?    ", alias["W1"] is params["W1"], shallow["W1"] is params["W1"], deep["W1"] is params["W1"])

params["W1"] -= 1                    # ① 배열 값을 in-place로 바꾸면
print("① W1:", alias["W1"], shallow["W1"], deep["W1"])

params["b1"] = np.array([9.0])       # ② dict 칸을 새 배열로 바꾸면
print("② b1:", alias["b1"], shallow["b1"], deep["b1"])

dict 자체가 같은 객체?   True False False
안의 W1이 같은 배열?     True True False
① W1: [0. 1.] [0. 1.] [1. 2.]
② b1: [9.] [0.] [0.]


* ①: 얕은 복사는 배열을 공유하므로 **같이 바뀐다.** 깊은 복사만 처음 값을 지킨다.
* ②: 얕은 복사는 dict 자체는 따로 가지고 있으므로, `params`의 칸을 바꿔도 영향이 없다.

```
params  (dict 1) ┌ 'W1' : ─┐
                 └ 'b1' : ─┼─┐
                           ├─┼──▶ 배열 [1. 2.]     ← 얕은 복사: 배열은 하나뿐 (공유)
shallow (dict 2) ┌ 'W1' : ─┘ │
                 └ 'b1' : ───┴──▶ 배열 [0.]
```

## 20.2. 숫자 배열 하나는 `.copy()`로 충분하다
숫자만 담긴 numpy 배열에는 다른 객체를 가리키는 참조가 없고 숫자가 그대로 들어 있다.
그래서 `arr.copy()`만으로 값이 모두 복사되고, 얕은 복사와 깊은 복사의 구별이 없다.

In [12]:
running_mean = np.array([0.12, 0.03])
saved = running_mean.copy()           # 새 메모리에 값을 복사한 새 배열
running_mean *= 0.5                   # 원래 배열을 in-place로 바꿔도
print("running_mean:", running_mean, " saved:", saved, " same:", saved is running_mean)

running_mean: [0.06  0.015]  saved: [0.12 0.03]  same: False


## 20.3. 실전: 가장 좋았던 epoch의 network를 저장해 두기
학습 중 검증 정확도가 가장 높을 때의 상태를 저장했다가 마지막에 되돌린다.
학습이 계속되면서 `params`의 배열들이 in-place로 바뀌므로, 저장본은 **깊은 복사**여야 한다.

```python
def get_state(net):
    state = {"params": copy.deepcopy(net.params), "bn": []}      # 배열이 든 dict -> deepcopy
    for layer in net.layers:
        if isinstance(layer, BatchNormalization):
            state["bn"].append((layer.running_mean.copy(), layer.running_var.copy()))  # 배열 하나 -> .copy()
    return state
```

PyTorch에서는 같은 일을 이렇게 한다. `state_dict()`는 model 안의 tensor를 복사하지 않고 그대로 가리키므로 `deepcopy`가 필요하다.
```python
best_state = copy.deepcopy(model.state_dict())     # 저장
model.load_state_dict(best_state)                  # 되돌리기
```

**정리**: 배열이 든 dict나 model은 `copy.deepcopy()`, 배열 하나는 `arr.copy()` (tensor는 `t.clone()`).

# 21. numpy의 `axis`와 `keepdims`

`sum`, `max`, `mean` 같은 함수에 `axis`를 주면 그 축 방향으로 계산하고, **그 축이 없어진다.**
`keepdims=True`를 주면 그 축을 **크기 1로 남겨 둔다.**

In [13]:
U = np.arange(6.).reshape(2, 3)
print(U)
print("U.max(axis=1)               :", U.max(axis=1), U.max(axis=1).shape)
print("U.max(axis=1, keepdims=True):", U.max(axis=1, keepdims=True).tolist(), U.max(axis=1, keepdims=True).shape)
print("U.sum(axis=0)               :", U.sum(axis=0), U.sum(axis=0).shape)

[[0. 1. 2.]
 [3. 4. 5.]]
U.max(axis=1)               : [2. 5.] (2,)
U.max(axis=1, keepdims=True): [[2.0], [5.0]] (2, 1)
U.sum(axis=0)               : [3. 5. 7.] (3,)


## 21.1. 왜 필요한가: broadcasting
numpy는 shape이 다른 두 배열을 계산할 때 **오른쪽 축부터** 맞춰 보고, 크기가 1인 축은 상대에 맞게 늘린다.
행마다 구한 값을 원래 배열의 각 행에서 빼려면 `(2, 3) - (2, 1)`이 되어야 한다.

In [14]:
print(U - U.max(axis=1, keepdims=True))        # (2,3) - (2,1): 각 행에서 그 행의 최댓값을 뺀다
try:
    U - U.max(axis=1)                          # (2,3) - (2,): 오른쪽 축 3과 2가 맞지 않는다
except ValueError as e:
    print("ValueError:", e)

[[-2. -1.  0.]
 [-2. -1.  0.]]
ValueError: operands could not be broadcast together with shapes (2,3) (2,) 


## 21.2. 더 위험한 경우: 행과 열의 수가 같을 때
`(3, 3) - (3,)`은 오류 없이 계산되지만 `(3,)`이 **행 방향으로** 늘어나서 엉뚱한 값을 뺀다. 오류가 나지 않으므로 알아채기 어렵다.

In [15]:
S = np.arange(9.).reshape(3, 3)
print("keepdims 없음 (틀림):\n", S - S.max(axis=1))
print("keepdims=True (맞음):\n", S - S.max(axis=1, keepdims=True))

keepdims 없음 (틀림):
 [[-2. -4. -6.]
 [ 1. -1. -3.]
 [ 4.  2.  0.]]
keepdims=True (맞음):
 [[-2. -1.  0.]
 [-2. -1.  0.]
 [-2. -1.  0.]]


**softmax** 에서 행마다 최댓값을 빼고 합으로 나누는 계산이 바로 이 경우이다.

In [16]:
def softmax(U):
    U = U - U.max(axis=1, keepdims=True)                      # overflow 방지
    return np.exp(U) / np.exp(U).sum(axis=1, keepdims=True)   # 행마다 합이 1

Y = softmax(np.array([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]]))
print(Y.round(4), Y.sum(axis=1))

[[0.09   0.2447 0.6652]
 [0.3333 0.3333 0.3333]] [1. 1.]


**규칙**: `axis=1`(행마다)로 줄인 값을 원래 배열과 다시 계산할 때는 `keepdims=True`가 필요하다.
`axis=0`(열마다)로 줄인 값은 결과 `(n,)`이 오른쪽 축과 맞으므로 없어도 되지만, 넣어 두면 헷갈리지 않는다.

# 22. 문자열 더 알아보기

## 22.1. f-string과 서식 지정자
`f"..."`는 **formatted string literal**(f-string)이다. `print`와는 상관없이, 그 자체로 **문자열을 만드는 식**이다.
`{식:서식}`의 `:` 뒤에 오는 부분을 **서식 지정자(format specifier)** 라고 한다. 값을 바꾸지 않고, **보여 주는 모양**만 정한다.

| 서식 | 예 | 결과 | 뜻 |
|---|---|---|---|
| `.4f` | `f"{0.98276:.4f}"` | `0.9828` | 소수점 아래 4자리 고정 소수점 |
| `2d` | `f"{3:2d}"` | `' 3'` | 정수, 최소 2칸 |
| `03d` | `f"{7:03d}"` | `007` | 빈칸을 0으로 채움 |
| `.2e` | `f"{0.000123:.2e}"` | `1.23e-04` | 지수 표기 |
| `.1%` | `f"{0.9828:.1%}"` | `98.3%` | 100을 곱해 퍼센트로 |
| `,` | `f"{110610:,}"` | `110,610` | 천 단위 쉼표 |
| `>8` | `f"{'acc':>8}"` | `'     acc'` | 8칸 안에서 오른쪽 정렬 |

In [17]:
acc, lr, n_params, epoch = 0.98276543, 0.000123, 110610, 3
print(f"epoch {epoch:2d} | acc {acc:.4f} | acc {acc:.1%} | lr {lr:.2e} | params {n_params:,}")
print(f"[{'acc':>8}] [{'acc':<8}] [{7:03d}]")

msg = f"val acc {acc:.4f}"           # print 밖에서도 쓸 수 있다: 그냥 문자열이다
fname = f"result_lr{lr}_ep{epoch:02d}.png"
print(msg, "|", fname, "|", type(msg))
print("acc itself is unchanged:", acc)

epoch  3 | acc 0.9828 | acc 98.3% | lr 1.23e-04 | params 110,610
[     acc] [acc     ] [007]
val acc 0.9828 | result_lr0.000123_ep03.png | <class 'str'>
acc itself is unchanged: 0.98276543


## 22.2. 문자열 접두사: `f`, `r`, `b`
| 접두사 | 이름 | 하는 일 | 타입 |
|---|---|---|---|
| `f"..."` | formatted | `{ }` 안의 값을 끼워 넣는다 | `str` |
| `r"..."` | raw | `\`를 특수 문자로 해석하지 않고 그대로 둔다 (Windows 경로, LaTeX) | `str` |
| `b"..."` | bytes | 글자가 아니라 바이트(0–255)의 나열 | `bytes` |

`f`와 `r`은 함께 쓸 수 있다 (`rf"..."`). 이때 LaTeX의 중괄호를 글자로 쓰려면 `{{ }}`처럼 두 번 쓴다.

In [18]:
print("normal:", "C:\new\test")          # \n은 줄바꿈, \t는 탭으로 해석된다
print("raw   :", r"C:\new\test")         # 그대로 남는다

lr = 0.01
print(rf"$\alpha = {lr}$,  $x^{{2}}$")   # matplotlib 제목에 쓸 수 있는 LaTeX 문자열

b = "딥러닝".encode("utf-8")              # str -> bytes
print(b, type(b), "->", b.decode("utf-8"))

normal: C:
ew	est
raw   : C:\new\test
$\alpha = 0.01$,  $x^{2}$
b'\xeb\x94\xa5\xeb\x9f\xac\xeb\x8b\x9d' <class 'bytes'> -> 딥러닝


# 23. `zip`과 unpacking 더 알아보기

## 23.1. `zip`
`zip`은 여러 list를 **같은 순서끼리 한 쌍씩** 묶어 tuple로 내놓는다 (옷의 지퍼처럼). 반복할 때 하나씩 내놓으므로, 한눈에 보려면 `list()`로 감싼다.

In [19]:
names  = ["Kim", "Lee", "Park"]
scores = [90, 85, 77]
print(list(zip(names, scores)))

for name, score in zip(names, scores):
    print(name, score)

print(list(zip([1, 2, 3], ["a", "b"])))     # 길이가 다르면 짧은 쪽에 맞춰 조용히 멈춘다
try:
    list(zip([1, 2, 3], ["a", "b"], strict=True))   # Python 3.10+: 길이가 다르면 오류
except ValueError as e:
    print("strict=True:", e)

[('Kim', 90), ('Lee', 85), ('Park', 77)]
Kim 90
Lee 85
Park 77
[(1, 'a'), (2, 'b')]
strict=True: zip() argument 2 is shorter than argument 1


## 23.2. unpacking
모양을 맞춰 이름을 적으면 tuple이나 list가 그 자리에서 풀린다. 이름이 받는 것은 역시 **참조**이며, 복사는 일어나지 않는다.

In [20]:
a, b = 1, 2
a, b = b, a                       # 두 값 바꾸기
print(a, b)

layers = ["BN1", "BN2"]
saved  = [(0.1, 1.0), (0.2, 0.9)]             # (mean, var) tuple들
for layer, (mean, var) in zip(layers, saved): # 괄호로 안쪽 tuple까지 푼다
    print(layer, mean, var)

first, *rest = [10, 20, 30, 40]               # *: 나머지를 list로 받는다
print(first, rest)

value, _ = (3.14, "unused")                   # _: 쓰지 않는 값에 붙이는 관례적인 이름
print(value)

2 1
BN1 0.1 1.0
BN2 0.2 0.9
10 [20, 30, 40]
3.14


# 24. 객체지향 프로그래밍 맛보기: 클래스와 상속

**객체지향 프로그래밍(object-oriented programming)** 은 데이터와 그 데이터를 다루는 함수를 하나의 **객체**로 묶고,
객체들이 협력하도록 프로그램을 짜는 방식이다. "지향(oriented)"은 "객체를 중심으로"라는 뜻이다.

* **클래스(class)**: 객체의 설계도. **속성(attribute)** 과 **메서드(method)** 를 정한다.
* `__init__`: 객체를 만들 때 실행되는 초기화 메서드. `self`는 만들어지는 객체 자신이다.
* **상속(inheritance)**: 기존 클래스를 물려받아 새 클래스를 만든다. `super().__init__()`은 부모 클래스의 `__init__`을 실행한다.
* 같은 이름의 메서드(`forward`)를 클래스마다 다르게 만들면, 쓰는 쪽은 종류를 따지지 않고 `layer.forward(x)`만 부르면 된다.

In [21]:
class Layer:                                  # 부모 클래스
    def __init__(self, name):
        self.name = name                      # 속성
    def forward(self, x):                     # 메서드 (자식이 다시 정의한다)
        raise NotImplementedError
    def __repr__(self):
        return f"{type(self).__name__}({self.name})"

class Scale(Layer):                           # Layer를 상속
    def __init__(self, name, a):
        super().__init__(name)                # 부모의 초기화를 먼저 실행
        self.a = a
    def forward(self, x):
        return self.a * x

class Relu(Layer):
    def forward(self, x):
        return np.maximum(0, x)

net = [Scale("s1", 2.0), Relu("r1"), Scale("s2", -1.0)]
x = np.array([-1.0, 0.5, 2.0])
for layer in net:                             # 종류를 따지지 않고 forward만 부른다
    x = layer.forward(x)
    print(layer, "->", x)
print(isinstance(net[0], Layer), isinstance(net[1], Scale))

Scale(s1) -> [-2.  1.  4.]
Relu(r1) -> [0. 1. 4.]
Scale(s2) -> [-0. -1. -4.]
True False


PyTorch의 모델도 같은 방식이다. `nn.Module`을 상속하고, `__init__`에서 층을 만들고, `forward`에서 계산 순서를 적는다.
`super().__init__()`을 먼저 불러야 PyTorch가 층과 parameter를 등록할 장부가 준비된다.

```python
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 100)
        self.fc2 = nn.Linear(100, 10)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))
```

# 25. 저장하고 불러오기

프로그램이 끝나면 메모리의 변수는 모두 사라진다. 결과를 남기려면 **파일로 저장**해야 한다.

| 방법 | 저장 형식 | 무엇을 저장 | 사람이 읽기 | 주로 쓰는 곳 |
|---|---|---|---|---|
| `json` | 텍스트 | dict, list, 숫자, 문자열, bool, None | 쉬움 | 실험 설정, 학습 기록(history), 결과 요약 |
| `pickle` | 바이너리 | 거의 모든 파이썬 객체 | 어려움 | 임시 저장, 파이썬끼리 객체 주고받기 |
| `np.save`, `np.savez` | 바이너리 (`.npy`, `.npz`) | numpy 배열 | 어려움 | 큰 배열, 가중치 |
| `torch.save` | 바이너리 (`.pt`) | tensor, `state_dict` | 어려움 | PyTorch 모델과 체크포인트 |

## 25.1. `json`: 사람이 읽을 수 있는 텍스트 파일

In [22]:
import json

config  = {"lr": 0.001, "epochs": 15, "hidden": [100, 100, 100, 100], "use_batchnorm": True}
history = {"loss": [0.2994, 0.1074, 0.0746], "val": [0.9654, 0.9704, 0.9738]}

with open("config.json", "w", encoding="utf-8") as f:     # "w": 쓰기 모드 (텍스트)
    json.dump({"config": config, "history": history}, f, indent=2)

with open("config.json", encoding="utf-8") as f:
    print(f.read())                                       # 메모장으로 열어도 읽을 수 있는 텍스트

with open("config.json", encoding="utf-8") as f:          # "r"(읽기)가 기본값
    loaded = json.load(f)
print(loaded["config"]["lr"], loaded["history"]["val"][-1], type(loaded["config"]["hidden"]))

{
  "config": {
    "lr": 0.001,
    "epochs": 15,
    "hidden": [
      100,
      100,
      100,
      100
    ],
    "use_batchnorm": true
  },
  "history": {
    "loss": [
      0.2994,
      0.1074,
      0.0746
    ],
    "val": [
      0.9654,
      0.9704,
      0.9738
    ]
  }
}


0.001 0.9738 <class 'list'>


`with open(...) as f:`는 블록이 끝나면 파일을 자동으로 닫아 준다.

json은 파이썬의 기본 타입만 저장할 수 있다. **numpy 배열은 바로 저장되지 않으므로** `.tolist()`로 list로 바꿔야 한다.
tuple은 list로 바뀌어 저장된다.

In [23]:
W = np.array([[1.0, 2.0], [3.0, 4.0]])
try:
    json.dumps({"W": W})
except TypeError as e:
    print("TypeError:", e)

s = json.dumps({"W": W.tolist(), "pair": (1, 2)})
print(s)
back = json.loads(s)
print(np.array(back["W"]), back["pair"])

TypeError: Object of type ndarray is not JSON serializable
{"W": [[1.0, 2.0], [3.0, 4.0]], "pair": [1, 2]}
[[1. 2.]
 [3. 4.]] [1, 2]


## 25.2. `pickle`: 파이썬 객체를 통째로
pickle은 numpy 배열이 든 dict, list, 직접 만든 클래스의 객체까지 그대로 저장한다. 바이너리 파일이므로 `"wb"`, `"rb"` 모드를 쓴다.

In [24]:
import pickle

state = {"params": {"W1": np.ones((2, 3)), "b1": np.zeros(3)},
         "bn": [(np.zeros(3), np.ones(3))],
         "epoch": 12}

with open("state.pkl", "wb") as f:        # "wb": 바이너리 쓰기
    pickle.dump(state, f)

with open("state.pkl", "rb") as f:        # "rb": 바이너리 읽기
    loaded = pickle.load(f)

print(loaded["epoch"], loaded["params"]["W1"].shape, type(loaded["bn"][0]))
print("same object as before saving?", loaded["params"]["W1"] is state["params"]["W1"])   # 파일에서 새로 만든 객체

12 (2, 3) <class 'tuple'>
same object as before saving? False


**주의**
* pickle 파일을 불러오면 그 안에 담긴 코드가 실행될 수 있다. **직접 만들었거나 믿을 수 있는 파일만** 열어야 한다.
* 파이썬 전용이고, 파이썬이나 라이브러리 버전이 크게 다르면 못 읽을 수 있다. 오래 보관할 결과는 json이나 `.npz`가 낫다.

## 25.3. numpy 배열: `np.save`, `np.savez`

In [25]:
np.save("W1.npy", W)                                  # 배열 하나
print(np.load("W1.npy"))

np.savez("params.npz", **state["params"])             # 여러 배열을 이름(key)과 함께
with np.load("params.npz") as d:                      # .npz는 파일을 열어 둔 채 읽으므로 with로 닫는다
    print(d.files, d["W1"].shape)

[[1. 2.]
 [3. 4.]]
['W1', 'b1'] (2, 3)


## 25.4. PyTorch: `torch.save`, `torch.load`
PyTorch는 모델 전체가 아니라 **`state_dict`(parameter와 BatchNorm의 이동평균)** 를 저장하는 것이 보통이다.
불러올 때는 **같은 구조의 모델을 먼저 만들고** 값을 채워 넣는다.

In [26]:
import torch
from torch import nn

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 2))
torch.save(model.state_dict(), "model.pt")                    # 가중치만 저장

model2 = nn.Sequential(nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 2))   # 같은 구조
model2.load_state_dict(torch.load("model.pt", weights_only=True))     # weights_only: 안전하게 읽기
x = torch.randn(1, 4)
print(torch.equal(model(x), model2(x)))

# 학습을 이어서 하려면 optimizer 상태와 epoch도 함께 저장한다 (체크포인트)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
torch.save({"epoch": 12, "model": model.state_dict(), "optimizer": optimizer.state_dict()}, "ckpt.pt")
ckpt = torch.load("ckpt.pt", weights_only=True)
print(list(ckpt.keys()))

True


['epoch', 'model', 'optimizer']


In [27]:
# 연습으로 만든 파일 정리
import os
for fn in ["config.json", "state.pkl", "W1.npy", "params.npz", "model.pt", "ckpt.pt"]:
    if os.path.exists(fn):
        os.remove(fn)
print("removed")

removed


# 26. 명령줄 인자: `sys.argv`, `argparse`

터미널에서 `python train.py --lr 1e-3`을 실행하면

| 부분 | 의미 |
|---|---|
| `python` | 파이썬 해석기 |
| `train.py` | 실행할 스크립트 |
| `--lr` | 옵션 이름 (learning rate) |
| `1e-3` | 옵션 값 $= 0.001$ |

명령줄에 적은 글자들을 **명령줄 인자(command-line argument)** 라고 하며, 파이썬은 이것을 공백으로 끊어 `sys.argv`라는 **문자열 list**에 넣는다.
`argparse`는 이 list를 규칙에 따라 분석해 주는 표준 모듈이다 (parse = 구문 분석).

1. `parser = argparse.ArgumentParser()`: 분석기 객체를 만든다.
2. `parser.add_argument("--lr", type=float, default=0.001)`: 규칙을 등록한다 (이름, 타입, 기본값, 설명).
3. `args = parser.parse_args()`: `sys.argv`를 분석한다. 값은 `args.lr`처럼 꺼낸다.

노트북에서는 명령줄이 없으므로, `parse_args`에 **list를 직접 넣어** 명령줄을 흉내 낼 수 있다.

In [28]:
import argparse

parser = argparse.ArgumentParser(description="MNIST training")
parser.add_argument("--lr", type=float, default=0.001, help="learning rate")
parser.add_argument("--epochs", type=int, default=15, help="number of epochs")

args = parser.parse_args(["--lr", "1e-2", "--epochs", "5"])   # python train.py --lr 1e-2 --epochs 5
print(args, "|", args.lr, type(args.lr))                      # 문자열 '1e-2'가 float 0.01이 된다

print(parser.parse_args([]))                                  # 아무것도 안 주면 기본값

args, rest = parser.parse_known_args(["--lr", "1e-2", "-f", "kernel.json"])
print(args, "| 모르는 인자:", rest)                            # parse_known_args: 모르는 인자는 따로 뺀다

parser.print_help()

Namespace(lr=0.01, epochs=5) | 0.01 <class 'float'>
Namespace(lr=0.001, epochs=15)
Namespace(lr=0.01, epochs=15) | 모르는 인자: ['-f', 'kernel.json']
usage: ipykernel_launcher.py [-h] [--lr LR] [--epochs EPOCHS]

MNIST training

options:
  -h, --help       show this help message and exit
  --lr LR          learning rate
  --epochs EPOCHS  number of epochs


## 26.1. 실제 스크립트로 실행해 보기
`%%writefile`로 셀 내용을 파일로 저장한 뒤, `!`로 터미널 명령을 실행한다 (Colab과 Jupyter에서 동작한다).
`{sys.executable}`은 지금 쓰고 있는 파이썬 해석기의 경로이다.

In [29]:
%%writefile train_demo.py
import sys
import argparse

parser = argparse.ArgumentParser(description="a demo of command-line arguments")
parser.add_argument("--lr", type=float, default=0.001, help="learning rate")
parser.add_argument("--epochs", type=int, default=15, help="number of epochs")
args = parser.parse_args()

print("sys.argv:", sys.argv)
print(f"training with lr = {args.lr}, epochs = {args.epochs}")

Writing train_demo.py


In [30]:
import sys
!{sys.executable} train_demo.py
!{sys.executable} train_demo.py --lr 1e-2 --epochs 5

sys.argv: ['train_demo.py']
training with lr = 0.001, epochs = 15


sys.argv: ['train_demo.py', '--lr', '1e-2', '--epochs', '5']
training with lr = 0.01, epochs = 5


이렇게 해 두면 코드를 고치지 않고 설정만 바꿔 여러 실험을 돌릴 수 있다. 예를 들어 Linux 서버(bash)에서는

```bash
for lr in 0.1 0.01 0.001; do python train.py --lr $lr; done
```

Windows PowerShell에서는

```powershell
foreach ($lr in 0.1, 0.01, 0.001) { python train.py --lr $lr }
```

설정이 많아지면 명령줄 대신 **설정 파일(json)** 을 만들어 `python train.py --config exp1.json`처럼 넘기기도 한다.
설정 파일 자체가 "이 실험을 어떻게 돌렸는지"의 기록이 된다.

In [31]:
import os
os.remove("train_demo.py")

# 27. 스크립트 실행과 노트북(커널) 실행

| | 터미널: `python 파일.py` | 노트북 / Interactive 창 (Jupyter 커널) |
|---|---|---|
| 프로세스 | 실행할 때 시작해서 끝나면 종료 | 커널을 끄거나 재시작할 때까지 유지 |
| 변수 | 실행이 끝나면 사라진다 | 셀 사이에서 계속 남는다 |
| 코드 전달 | 파일 전체를 위에서 아래로 한 번 | 셀 단위로 여러 번, 순서도 자유 |
| 파이썬 해석기 | 같다 | 같다 |

* **Jupyter 커널(kernel)** 은 코드를 받아 실행하고 결과를 돌려주며 다음 코드를 기다리는 파이썬 프로세스이다. 운영체제의 커널과는 이름만 같다.
* Jupyter는 커널, 화면(브라우저의 Jupyter, VS Code, Spyder 콘솔), 둘이 주고받는 규칙과 `.ipynb` 형식으로 이루어진다. Colab은 커널이 Google 서버에서 돌아간다.
* 터미널에서도 `python -i 파일.py`로 실행하면, 끝난 뒤 `>>>` 프롬프트가 남아 변수를 볼 수 있다.
* VS Code에서는 `.py` 파일에 `# %%`를 넣어 셀로 나누면, 같은 파일을 셀 단위(Interactive 창)로도, 터미널에서 통째로도 실행할 수 있다.

**주의**: 커널에는 예전에 만든 변수가 남아 있으므로, 코드에서 지운 변수가 여전히 살아 있어 "내 컴퓨터에서는 되는" 일이 생긴다.
코드를 제출하거나 나눠 주기 전에는 **커널을 재시작하고 처음부터 모두 실행**(Restart & Run all)해 보자.

# 28. 연습문제

1. 다음 코드의 출력을 예상한 뒤 실행해 확인하라.
   ```python
   a = np.zeros(3); b = a; c = a.copy()
   a += 1
   print(a, b, c)
   ```
2. `x = 300; y = x; x += 1` 뒤에 `y`의 값은? `x`와 `y`가 같은 객체를 가리키는가? 이유를 18절과 연결하여 설명하라.
3. `d = {"W": np.ones(2)}`에 대하여 `s = copy.copy(d)`, `t = copy.deepcopy(d)`를 만든 뒤 `d["W"] *= 3`을 실행하면 `s["W"]`와 `t["W"]`는 각각 얼마인가?
4. `X`의 shape이 `(100, 10)`일 때 `X.mean(axis=1)`과 `X.mean(axis=1, keepdims=True)`의 shape은? 각 행에서 그 행의 평균을 빼려면 어느 것을 써야 하는가?
5. `f"{0.000512:.1e}"`, `f"{0.97345:.2%}"`, `f"{42:05d}"`의 결과를 쓰라.
6. 학습 기록 `history = {"loss": [...], "val": [...]}`를 json 파일로 저장하고 다시 불러오는 코드를 쓰라. `history`에 numpy 배열이 들어 있다면 무엇을 바꿔야 하는가?
7. `--batch-size`(정수, 기본값 100) 옵션을 26절의 parser에 추가하고, `parse_args(["--batch-size", "32"])`로 확인하라. (값은 `args.batch_size`로 꺼낸다.)

In [32]:
# 연습문제 풀이 공간